<a href="https://colab.research.google.com/github/grsart/BiomolComp/blob/main/P10/Metagenomica16S_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Preparação

Nesta prática vamos usar o software **QIIME 2** (versão 2024.2) para analisar dados de sequenciamento do gene 16S rRNA ([documentação desta versão](https://docs.qiime2.org/2024.2/); a documentação atual fica em [amplicon-docs.qiime2.org](https://amplicon-docs.qiime2.org/)).

Aqui, vamos instalá-lo de uma forma que permite a execução no Google Colab.

Vamos nos basear em dois repositórios do GitHub:
1. [Gibbons-Lab/isb_course_2024](https://github.com/Gibbons-Lab/isb_course_2024) (dados e roteiro da prática)
2. [josruirod/QIIME2_Read_Preprocessing_FGCZ](https://github.com/josruirod/QIIME2_Read_Preprocessing_FGCZ) (instalação do QIIME 2 no Colab)

Os dados são um subconjunto do estudo de **Hill-Burns et al. (2017)**, *Parkinson's disease and Parkinson's disease medications have distinct signatures of the gut microbiome*, Movement Disorders 32(5): 739–749.

> ⏱️ **Rode as duas próximas células de código logo no início da aula.** A instalação leva cerca de 15 minutos e pode correr enquanto acompanhamos a parte teórica.

In [ ]:
!git clone https://github.com/gibbons-lab/isb_course_2024 materials

!git clone https://github.com/josruirod/QIIME2_Read_Preprocessing_FGCZ.git qiime2_installation
# fixa a versão do instalador testada nesta disciplina (commit de maio/2025)
!git -C qiime2_installation checkout -q d9d7d3d

Durante esta prática será muito útil navegar pelo diretório do Colab. Para visualizá-lo, basta clicar no ícone 📁 na barra do lado esquerdo.

PS: usamos `%cd` em vez de `!cd` para mudar de pasta. O `%cd` muda a pasta de trabalho de forma **permanente** para as próximas células; o `!cd` só vale dentro daquela mesma linha de comando.

O primeiro passo é instalar o QIIME 2. É esperado que demore entre 10 e 15 minutos.

In [ ]:
%cd /content/qiime2_installation

!pip uninstall -y bcrypt
!pip install bcrypt
%run setup_qiime2

Confira se a instalação funcionou: o comando abaixo deve listar a versão do QIIME 2 e os *plugins* instalados.

In [ ]:
!qiime info

In [ ]:
%cd /content/materials

---
# Importando os dados

O QIIME 2 é uma ferramenta poderosa que, a partir dos dados brutos de sequenciamento, permite inferir a composição e a diversidade das comunidades microbianas.

O *workflow* abaixo foi obtido do GitHub do Gibbons Lab e descreve o que será feito nesta prática:

![our workflow](https://github.com/Gibbons-Lab/isb_course_2024/raw/main/docs/16S/assets/steps.png)

Precisamos de três arquivos:

1. **Arquivos brutos de sequenciamento** (.fastq): sequência + pontuação de qualidade de cada base.

2. **Arquivo de metadados** (.tsv): informações sobre as condições experimentais e as amostras. Neste caso, inclui:
- Estado de doença
- Idade
- Sexo
- Localização geográfica
- Uso de medicamentos
- Dieta
- Sintomas ...

3. **Arquivo de manifesto** (.tsv): conecta cada amostra (pelo seu identificador) ao arquivo de sequências correspondente.

As leituras são *single-end* e cobrem a região **V4** do 16S rRNA (primers 515F–806R).

Vamos visualizar o arquivo `manifest.tsv`.

**Q1 - Quantos arquivos serão utilizados durante a prática?**

Esses arquivos representam as amostras coletadas que serão analisadas.

In [ ]:
import pandas as pd
manifest = pd.read_csv('data/manifest.tsv', sep='\t')
manifest

Verificando o arquivo `metadata.tsv`:

**Q2 - Quantas características descrevem as amostras? Quantas amostras são de pacientes com doença de Parkinson e quantas não? Compare os dois grupos quanto ao uso de medicamentos (em especial `carbidopa_levodopa`), ao sexo, à cidade de coleta e ao uso de antibióticos nos últimos 3 meses (`p3m_antibiotics_bool`). Se encontrarmos diferenças no microbioma entre os grupos, poderemos atribuí-las à doença?**

In [ ]:
metadata = pd.read_csv('data/metadata.tsv', sep='\t')
metadata

## FASTQ para Artifact

Vamos importar os dados para dentro do QIIME 2, usando os arquivos FASTQ e o manifesto.

O QIIME 2 tem a sua própria forma de armazenar dados, chamada de *artifacts*, com extensão **.qza**.

In [ ]:
!qiime tools import \
  --type 'SampleData[SequencesWithQuality]' \
  --input-path data/manifest.tsv \
  --output-path sequences.qza \
  --input-format SingleEndFastqManifestPhred33V2

Usamos o *plugin* `tools` do QIIME 2 para importar os dados. Os outros argumentos indicam o arquivo de entrada (manifesto), o formato das sequências e onde o arquivo de saída será salvo.

## Artifact para visualização

É sempre útil converter os *artifacts* em um formato de visualização, **.qzv**, que podemos explorar de forma interativa em um visualizador na web.

In [ ]:
!qiime demux summarize \
  --i-data sequences.qza \
  --o-visualization qualities.qzv

O arquivo .qzv gerado pode ser visualizado no site **https://view.qiime2.org**. Primeiro, faça o download do arquivo: clique no ícone de pasta do lado esquerdo, navegue até a pasta `materials` e selecione o arquivo `qualities.qzv`. Ao clicar nos três pontos verticais, aparecerá a opção *Download*. Depois, arraste o arquivo para o visualizador e abra a aba *Interactive Quality Plot*.

**Q3 - Qual é o comprimento das leituras? Em que posição a qualidade mediana começa a cair? Na próxima etapa vamos truncar todas as leituras em 150 pb: essa escolha parece razoável? O que aconteceria se truncássemos muito antes ou muito depois?**

---
# Filtro de sequências por parâmetros de qualidade

Antes de usar os dados, precisamos remover os ruídos (erros de sequenciamento). Para isso, vamos usar o *plugin* `DADA2`, que executa os seguintes passos:

1. Filtrar e truncar as leituras
2. Encontrar o conjunto mais provável de sequências únicas nas amostras (ASVs, *amplicon sequence variants*)
3. Remover quimeras
4. Contar a abundância de cada ASV em cada amostra

Este comando leva cerca de 2 minutos.

In [ ]:
!qiime dada2 denoise-single \
    --i-demultiplexed-seqs sequences.qza \
    --p-trunc-len 150 \
    --p-n-threads 2 \
    --output-dir dada --verbose

**EXECUTE O COMANDO ABAIXO SOMENTE SE A CÉLULA ACIMA DER ERRO**

O próximo comando recupera um arquivo de *backup* para seguir a prática, em caso de erro.

In [ ]:
# só copia o backup se a pasta dada não existir (ou seja, se o comando anterior falhou)
![ -d dada ] || cp -r treasure_chest/dada .

Se você navegar pela pasta `dada`, verá que foram gerados três arquivos:
- `table.qza` - a **tabela de abundância**: quantas leituras de cada ASV foram encontradas em cada amostra
- `representative_sequences.qza` - a **sequência de DNA** de cada ASV
- `denoising_stats.qza` - as **estatísticas** de quantas leituras sobraram em cada etapa da remoção de ruídos

Vamos começar pelas estatísticas de remoção de ruídos.

Lembrando: os arquivos .qza não podem ser lidos diretamente, mas podem ser convertidos em visualização (.qzv) pelo *plugin* `metadata`, com a ação `tabulate`.

In [ ]:
!qiime metadata tabulate \
    --m-input-file dada/denoising_stats.qza \
    --o-visualization dada/denoising_stats.qzv

Faça o download do arquivo `dada/denoising_stats.qzv` e abra no [QIIME 2 Viewer](https://view.qiime2.org/).

**Q4 - Qual a porcentagem de leituras que passou pela etapa de filtragem? Qual a porcentagem de leituras não quiméricas? Como esses valores variam entre as amostras?**

É normal perder entre 5 e 25% das leituras por causa da formação de quimeras. Use essa informação para discutir a resposta da Q4.

## Quantas leituras sobraram em cada amostra?

Mais adiante, vamos precisar escolher uma **profundidade de rarefação** (número de leituras sorteadas por amostra). Para isso, primeiro precisamos saber quantas leituras cada amostra tem depois do DADA2.

In [ ]:
!qiime feature-table summarize \
    --i-table dada/table.qza \
    --m-sample-metadata-file data/metadata.tsv \
    --o-visualization dada/table.qzv

O arquivo `dada/table.qzv` pode ser aberto no visualizador (aba *Interactive Sample Detail*). Também podemos ver os mesmos números aqui mesmo, exportando a tabela:

In [ ]:
!qiime tools export --input-path dada/table.qza --output-path dada/table_exportada
!biom convert -i dada/table_exportada/feature-table.biom -o dada/table.tsv --to-tsv

asvs = pd.read_table('dada/table.tsv', skiprows=1, index_col=0)
leituras_por_amostra = asvs.sum().sort_values()
print('Número de ASVs:', asvs.shape[0])
leituras_por_amostra

---
# Filogenia

Agora que temos a tabela de abundância dos ASVs, podemos calcular métricas de diversidade.

### Gerando a árvore filogenética dos ASVs com o módulo `phylogeny` do QIIME 2

Algumas métricas de diversidade (Faith PD, UniFrac) usam a **distância filogenética** entre os ASVs. Por isso, vamos construir uma árvore filogenética: os ASVs são alinhados com o MAFFT, as posições muito variáveis são mascaradas e a árvore é inferida com o FastTree e enraizada no ponto médio.

In [ ]:
!qiime phylogeny align-to-tree-mafft-fasttree \
    --i-sequences dada/representative_sequences.qza \
    --output-dir tree

Para visualizar a árvore, vamos usar o *plugin* `empress` do QIIME 2. Mais informações estão na [documentação](https://github.com/biocore/empress).

In [ ]:
!qiime empress tree-plot \
    --i-tree tree/rooted_tree.qza \
    --o-visualization tree/empress.qzv

Abra `tree/empress.qzv` no visualizador. Por enquanto, as folhas da árvore são apenas identificadores de ASVs; no final da prática, vamos colori-la com a taxonomia e os metadados.

---
# Diversidade

## Parte 1: escolhendo a profundidade de rarefação

As amostras têm números de leituras diferentes, e a diversidade observada cresce com o esforço amostral. Para comparar amostras de forma justa, vamos **rarear**: sortear aleatoriamente o mesmo número de leituras de cada amostra. Amostras com menos leituras do que a profundidade escolhida são descartadas.

A curva de rarefação mostra como a diversidade cresce à medida que sorteamos mais leituras:

In [ ]:
PROFUNDIDADE_MAX = int(leituras_por_amostra.max())

!qiime diversity alpha-rarefaction \
    --i-table dada/table.qza \
    --i-phylogeny tree/rooted_tree.qza \
    --p-max-depth {PROFUNDIDADE_MAX} \
    --m-metadata-file data/metadata.tsv \
    --o-visualization rarefacao.qzv

Abra `rarefacao.qzv` no visualizador.

**Q5 - A partir de qual profundidade as curvas de diversidade se estabilizam? Usando a tabela de leituras por amostra (acima), quantas amostras seriam descartadas com uma profundidade de 5.000? E de 15.000? Qual é o compromisso envolvido nessa escolha?**

## Parte 2: calculando as métricas

Dentro do *plugin* `diversity` existe a ação `core-metrics-phylogenetic`, que rareia a tabela e calcula várias métricas de diversidade, filogenéticas ou não.

Neste ponto, usamos os metadados para saber de quais grupos vêm as amostras.

Como a rarefação é **aleatória**, os resultados podem ficar ligeiramente diferentes entre vocês.

In [ ]:
PROFUNDIDADE = 5000   # altere aqui se a sua resposta da Q5 indicar outro valor

!qiime diversity core-metrics-phylogenetic \
    --i-table dada/table.qza \
    --i-phylogeny tree/rooted_tree.qza \
    --p-sampling-depth {PROFUNDIDADE} \
    --m-metadata-file data/metadata.tsv \
    --output-dir diversity

Todas as métricas de diversidade ficam salvas na pasta `diversity`. Para saber como são calculadas e como interpretá-las, veja este [post do fórum do QIIME 2](https://forum.qiime2.org/t/alpha-and-beta-diversity-explanations-and-commands/2282).

Agora entra a estatística: queremos saber se a diversidade difere ***significativamente*** entre as amostras de pacientes com Parkinson e controles, ou segundo alguma outra variável.

### Diversidade alfa: riqueza e equitabilidade *dentro* de cada amostra

Vamos comparar a diversidade alfa entre os grupos usando três métricas que medem coisas diferentes:

- **Observed features**: riqueza, ou seja, quantos ASVs diferentes há na amostra
- **Shannon**: combina riqueza e equitabilidade (quão uniformemente as leituras se distribuem entre os ASVs)
- **Faith PD**: diversidade filogenética, a soma dos ramos da árvore ocupados pelos ASVs da amostra

Uma diversidade alfa maior costuma estar associada a um intestino saudável, mas, sozinha, ela não é um indicador de saúde ou doença.

O teste usado é o de Kruskal-Wallis, aplicado a cada coluna categórica dos metadados.

In [ ]:
!qiime diversity alpha-group-significance \
    --i-alpha-diversity diversity/observed_features_vector.qza \
    --m-metadata-file data/metadata.tsv \
    --o-visualization diversity/alpha_groups_observed_features.qzv

!qiime diversity alpha-group-significance \
    --i-alpha-diversity diversity/shannon_vector.qza \
    --m-metadata-file data/metadata.tsv \
    --o-visualization diversity/alpha_groups_shannon.qzv

!qiime diversity alpha-group-significance \
    --i-alpha-diversity diversity/faith_pd_vector.qza \
    --m-metadata-file data/metadata.tsv \
    --o-visualization diversity/alpha_groups_faith_pd.qzv

Faça o download dos três arquivos `diversity/alpha_groups_*.qzv` e abra no [QIIME 2 Viewer](https://view.qiime2.org).

**Q6 - Como a diversidade alfa se comporta nas amostras? Há algum padrão com a condição de saúde? As três métricas contam a mesma história? Com 5 amostras por grupo, que poder estatístico esperamos ter?**

### Diversidade beta: distância entre amostras

Para a diversidade beta, vamos usar a distância **weighted UniFrac**, que leva em conta a filogenia e a abundância dos ASVs (a *unweighted* considera só presença/ausência).

Podemos avaliar se os grupos se separam significativamente com o **teste PERMANOVA** (ação `adonis`):

In [ ]:
!qiime diversity adonis \
    --i-distance-matrix diversity/weighted_unifrac_distance_matrix.qza \
    --m-metadata-file data/metadata.tsv \
    --p-formula "parkinson_disease" \
    --p-n-jobs 2 \
    --o-visualization diversity/permanova.qzv

Faça o download e abra no [QIIME 2 Viewer](https://view.qiime2.org):
- `diversity/permanova.qzv`: resultado do teste (veja as colunas **R2**, a fração da variação explicada, e **Pr(>F)**, o valor de p);
- `diversity/weighted_unifrac_emperor.qzv`: PCoA da mesma distância. Pinte os pontos por `parkinson_disease` e depois por `sex`, `location` e `carbidopa_levodopa`;
- para comparar, `diversity/unweighted_unifrac_emperor.qzv`.

**Q7 - Esses resultados são informativos? Discuta levando em conta o tamanho da amostra e o que você encontrou na Q2: se o PERMANOVA indicar diferença entre os grupos, ela pode ser atribuída à doença, ao medicamento, ao sexo ou à cidade de coleta? Como uma amostra maior mudaria essa análise?**

---
# O estudo completo: controlando fatores de confusão

As 10 amostras que analisamos até aqui são um pequeno subconjunto do estudo. O Gibbons Lab já processou o estudo completo com os mesmos passos (DADA2, árvore, `core-metrics-phylogenetic`), e os resultados estão na pasta `treasure_chest/complete_study`. Vamos copiá-los:

In [ ]:
![ -d complete_study ] || cp -r treasure_chest/complete_study .

In [ ]:
complete_metadata = pd.read_csv('complete_study/complete_metadata.tsv', sep='\t')
print('Amostras no estudo completo:', len(complete_metadata))
print()
print(complete_metadata['parkinson_disease'].value_counts())
print()
# uso de levodopa em cada grupo (NaN = sem informação)
pd.crosstab(complete_metadata['parkinson_disease'],
            complete_metadata['carbidopa_levodopa'].fillna('sem dado'))

Primeiro, repetimos as análises de diversidade alfa e beta com todas as amostras:

In [ ]:
!qiime diversity alpha-group-significance \
    --i-alpha-diversity complete_study/diversity/shannon_vector.qza \
    --m-metadata-file complete_study/complete_metadata.tsv \
    --o-visualization complete_study/diversity/alpha_groups.qzv

!qiime diversity adonis \
    --i-distance-matrix complete_study/diversity/weighted_unifrac_distance_matrix.qza \
    --m-metadata-file complete_study/complete_metadata.tsv \
    --p-formula "parkinson_disease" \
    --p-n-jobs 2 \
    --o-visualization complete_study/diversity/permanova_PD.qzv

Agora vamos incluir as possíveis **variáveis de confusão** no PERMANOVA. Assim, o efeito da doença é avaliado *depois* de descontada a variação explicada por sexo, idade, IMC, cidade e medicamentos.

O PERMANOVA não aceita valores faltantes, então antes removemos da matriz de distâncias as amostras sem informação nessas variáveis:

In [ ]:
!qiime diversity filter-distance-matrix \
    --i-distance-matrix complete_study/diversity/weighted_unifrac_distance_matrix.qza \
    --m-metadata-file complete_study/complete_metadata.tsv \
    --p-where "[bmi] IS NOT NULL AND [comt_inhibitor] IS NOT NULL AND [carbidopa_levodopa] IS NOT NULL AND [anticholinergic] IS NOT NULL" \
    --o-filtered-distance-matrix complete_study/diversity/weighted_unifrac_distance_matrix_no_NA.qza

!qiime diversity adonis \
    --i-distance-matrix complete_study/diversity/weighted_unifrac_distance_matrix_no_NA.qza \
    --m-metadata-file complete_study/complete_metadata.tsv \
    --p-formula "parkinson_disease + sex + age + bmi + location + comt_inhibitor + carbidopa_levodopa + anticholinergic" \
    --p-n-jobs 2 \
    --o-visualization complete_study/diversity/permanova_with_confounders.qzv

Abra `complete_study/diversity/alpha_groups.qzv`, `permanova_PD.qzv` e `permanova_with_confounders.qzv`.

**Q8 - Com o estudo completo, a diferença entre pacientes com Parkinson e controles fica significativa (alfa e beta)? Depois de incluir as covariáveis, a doença continua explicando parte da variação? Quanto (R2)? Quais outras variáveis explicam variação significativa? No estudo completo ainda existe confundimento total entre doença e levodopa? Que parte do efeito você atribuiria à doença e que parte ao medicamento?**

*Atenção:* no `adonis`, a ordem dos termos na fórmula importa (somas de quadrados sequenciais). Cada termo é avaliado depois dos que vêm antes dele.

---
# Classificação taxonômica

### Construção de uma tabela de contagem taxonômica

Voltando às nossas 10 amostras: a pergunta agora é **quais micro-organismos** estão em cada amostra. Vamos usar um classificador bayesiano *naive* pré-treinado. Outros classificadores podem ser encontrados em https://docs.qiime2.org/2024.2/data-resources/.

Especificamente, vamos usar um classificador treinado com sequências do NCBI RefSeq recortadas na região V4 do 16S (primers 515F–806R), que é a região que foi sequenciada. Um classificador treinado em outra região classificaria mal as nossas leituras.

In [ ]:
# classifica os ASVs das nossas 10 amostras
!qiime feature-classifier classify-sklearn \
    --i-reads dada/representative_sequences.qza \
    --i-classifier ncbi-refseq-genus-515f-806r.qza \
    --p-n-jobs 2 \
    --o-classification taxa.qza

Com as sequências classificadas, podemos visualizar a composição taxonômica das amostras:

In [ ]:
!qiime taxa barplot \
    --i-table dada/table.qza \
    --i-taxonomy taxa.qza \
    --m-metadata-file data/metadata.tsv \
    --o-visualization taxa_barplot.qzv

Abra `taxa_barplot.qzv` no visualizador. Escolha o nível taxonômico 2 (filo) e depois o 6 (gênero), e ordene as amostras por `parkinson_disease`.

A tabela `table.qza` contém as abundâncias dos ASVs, e o arquivo `taxa.qza` contém a classificação taxonômica de cada ASV. Vamos usar as duas informações para somar as abundâncias por **gênero** (nível 6).

In [ ]:
!qiime taxa collapse \
    --i-table dada/table.qza \
    --i-taxonomy taxa.qza \
    --p-level 6 \
    --o-collapsed-table genus.qza

### Exportando a tabela de gêneros

Novamente, precisamos converter o formato da tabela gerada pelo QIIME 2 para que possa ser lida por outros programas. Vamos converter para o formato .biom e depois para .tsv.

In [ ]:
!qiime tools export \
    --input-path genus.qza \
    --output-path exported

!biom convert -i exported/feature-table.biom -o genus.tsv --to-tsv

Vamos dar uma olhada nos resultados:

In [ ]:
abundances = pd.read_table("genus.tsv", skiprows=1, index_col=0)
abundances

Para facilitar a interpretação, vamos fazer um mapa de calor com a biblioteca `seaborn`.

Antes, transformamos as contagens. Os dados de sequenciamento são **composicionais**: cada amostra tem um total de leituras arbitrário, e só as proporções entre os táxons têm significado. Por isso, usamos a transformação **CLR** (*centered log-ratio*):

$$\mathrm{CLR}(x_i) = \log x_i - \frac{1}{D}\sum_{j=1}^{D} \log x_j$$

ou seja, o log de cada gênero menos a média dos logs (o log da média **geométrica**) dos $D$ gêneros da amostra. Somamos 0,5 às contagens para evitar $\log 0$.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

In [ ]:
generos = abundances.copy()
generos.index = generos.index.str.split(";").str[5]   # usa só o nome do gênero

# CLR calculado em cada amostra (coluna), com todos os gêneros
log_abund = np.log(generos + 0.5)
clr = log_abund - log_abund.mean(axis=0)

# remove gêneros não classificados e mantém os 50 mais abundantes
classificado = generos.index.notna() & ~generos.index.isin(["g__", "__"])
clr = clr[classificado]
top50 = generos[classificado].sum(axis=1).to_numpy().argsort()[::-1][:50]
transformed = clr.iloc[top50]

# barra lateral de cores com o grupo de cada amostra
cores_grupo = {'Yes': '#d62728', 'No': '#1f77b4'}
cores = metadata.set_index('id').loc[transformed.columns, 'parkinson_disease'].map(cores_grupo)

g = sns.clustermap(transformed.T, cmap="magma", xticklabels=True, row_colors=cores, figsize=(16, 6),
                   cbar_kws={'label': 'CLR'})
g.fig.legend(handles=[Patch(color=cores_grupo['Yes'], label='Parkinson'),
                      Patch(color=cores_grupo['No'], label='Controle')],
             loc='upper right')
plt.show()

Cada linha é uma amostra (a barra colorida à esquerda indica o grupo) e cada coluna é um gênero bacteriano. As cores mostram o valor **CLR**: valores positivos indicam que o gênero está acima da média (geométrica) da amostra, e valores negativos, abaixo. As árvores (dendrogramas) agrupam amostras e gêneros com perfis parecidos.

**Q9 - Existe algum padrão notável nas amostras? O agrupamento hierárquico separa os pacientes com Parkinson dos controles? Quais gêneros dominam a maioria das amostras?**

---
# Abundância diferencial no estudo completo

Quais gêneros são mais (ou menos) abundantes em pacientes com Parkinson? Com 10 amostras não há poder estatístico para isso, então vamos usar a tabela de gêneros do estudo completo.

Para cada gênero presente em pelo menos 10% das amostras, comparamos os valores CLR dos dois grupos com um **teste t de Welch**. Como fazemos centenas de testes, corrigimos os valores de p para **múltiplas comparações** pelo método de Benjamini-Hochberg (FDR); o valor corrigido é chamado de *q*.

In [ ]:
from scipy import stats
from statsmodels.stats.multitest import multipletests

completo = pd.read_table('complete_study/genus.tsv', skiprows=1, index_col=0)
meta_c = complete_metadata.set_index('id')
amostras = completo.columns.intersection(meta_c.index)
completo = completo[amostras]
grupo = meta_c.loc[amostras, 'parkinson_disease']
print('Amostras com dados de gênero:', len(amostras))
print(grupo.value_counts())

log_c = np.log(completo + 0.5)
clr_c = log_c - log_c.mean(axis=0)

prevalente = (completo > 0).mean(axis=1) >= 0.10
resultados = []
for linhagem, x in clr_c[prevalente].iterrows():
    pd_vals, hc_vals = x[grupo == 'Yes'], x[grupo == 'No']
    t, p = stats.ttest_ind(pd_vals, hc_vals, equal_var=False)
    resultados.append({'genero': linhagem.split(';')[-1],
                       'diferenca_CLR': pd_vals.mean() - hc_vals.mean(),
                       'p': p, 'linhagem': linhagem})

res = pd.DataFrame(resultados).dropna(subset=['p'])
res['q'] = multipletests(res['p'], method='fdr_bh')[1]
res = res.sort_values('q')
print('Gêneros testados:', len(res), '| com q < 0,1:', (res['q'] < 0.1).sum())
res.head(15)

Vamos olhar a distribuição dos 4 gêneros com menor *q*:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (_, linha) in zip(axes, res.head(4).iterrows()):
    dados = pd.DataFrame({'grupo': grupo, 'CLR': clr_c.loc[linha['linhagem']]})
    sns.boxplot(data=dados, x='grupo', y='CLR', ax=ax, order=['No', 'Yes'], showfliers=False)
    sns.stripplot(data=dados, x='grupo', y='CLR', ax=ax, order=['No', 'Yes'], color='k', size=3, alpha=0.5)
    ax.set_title(f"{linha['genero']}\nq = {linha['q']:.2g}")
    ax.set_xticks([0, 1], ['Controle', 'Parkinson'])
    ax.set_xlabel('')
plt.tight_layout()
plt.show()

**Q10 - Quais gêneros diferem entre os grupos após a correção para múltiplos testes (q < 0,1)? Eles estão aumentados ou diminuídos nos pacientes com Parkinson? Compare com os resultados de Hill-Burns et al. (2017). Considerando o que você viu na Q8, essas diferenças podem ser atribuídas à doença?**

*Para pensar:* por que não basta comparar as abundâncias relativas (proporções) diretamente? E por que não usamos simplesmente p < 0,05 em cada um dos testes?

---
# (Opcional) A árvore com taxonomia e metadados

Agora que temos a taxonomia, podemos voltar à árvore filogenética e colori-la. O `empress community-plot` combina a árvore, a tabela de abundâncias, os metadados das amostras e a taxonomia dos ASVs:

In [ ]:
!qiime empress community-plot \
    --i-tree tree/rooted_tree.qza \
    --i-feature-table dada/table.qza \
    --m-sample-metadata-file data/metadata.tsv \
    --m-feature-metadata-file taxa.qza \
    --o-visualization tree/empress_comunidade.qzv

Abra `tree/empress_comunidade.qzv`. Em *Feature Metadata Coloring*, pinte a árvore por `Level 2` (filo) e depois por `Level 5` (família). Em *Sample Metadata Coloring*, pinte por `parkinson_disease`.

**Q11 - Os filos formam clados na árvore? Existem ramos encontrados só em pacientes com Parkinson ou só em controles? Como isso se relaciona com a diferença entre UniFrac weighted e unweighted?**